In [ ]:
!pip install plotly --upgrade
!pip install kaleido --upgrade

In [ ]:
!plotly_get_chrome


Plotly will install a copy of Google Chrome to be used for generating static images of plots.
Chrome will be installed at: None
Do you want to proceed? [y/n] y
Installing Chrome for Plotly...
Chrome installed successfully.
The Chrome executable is now located at: /root/.local/share/choreographer/deps/chrome-linux64/chrome


In [ ]:
!pip show plotly kaleido

Name: plotly
Version: 6.9.0
Summary: An open-source interactive data visualization library for Python
Home-page: https://plotly.com/python/
Author: 
Author-email: Chris P <chris@plot.ly>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: narwhals, packaging
Required-by: cufflinks, geemap
---
Name: kaleido
Version: 1.3.0
Summary: Plotly graph export library
Home-page: https://github.com/plotly/kaleido
Author: 
Author-email: Andrew Pikul <ajpikul@gmail.com>, Neyberson Atencio <neyberatencio@gmail.com>
License: The MIT License (MIT)

Copyright (c) Plotly, Inc

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the follow

In [ ]:
import importlib.metadata as md

print("Plotly :", md.version("plotly"))
print("Kaleido:", md.version("kaleido"))

Plotly : 6.9.0
Kaleido: 1.3.0


In [ ]:
import pandas as pd
import kaleido
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/CRS_Revision_Code"
file_path = os.path.join(BASE_DIR, "outputfiles/csvs/df_ssp2_clean.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_path

'/content/drive/MyDrive/CRS_Revision_Code/outputfiles/csvs/df_ssp2_clean.csv'

In [ ]:
df_0 = pd.read_csv(file_path, index_col= 0)

In [ ]:
# Rename columns
df_0.columns = df_0.columns.str.replace("Burden", "Load", regex=False)

# Replace in string columns only
load_cols = ['RBUV_Load_2050', 'RBUV_Load_2100', 'RL_Load_2050', 'RL_Load_2100']
df_0[load_cols] = df_0[load_cols].replace("Burden", "Load", regex=True)

In [ ]:
# df_0

In [ ]:
volm3_bins = [0, 250, 500, 1000, 5000, 400000]
volm3_labels = ['0-250 m³', '250-500 m³', '500-1000 m³', '1000-5000 m³', '5000+ m³']
len_m_bins = [0, 5, 20, 50, 100, 15000]
len_m_labels = ['0-5 m', '5-20 m', '20-50 m', '50-100 m', '100+ m']
df_0['range_volm3_2020'] = pd.cut(df_0['volume_m3_perCap_2020_ssp2'], bins = volm3_bins, labels = volm3_labels, right=True)
df_0['range_volm3_2050'] = pd.cut(df_0['volume_m3_perCap_2050_ssp2'], bins = volm3_bins, labels = volm3_labels, right=True)
df_0['range_volm3_2100'] = pd.cut(df_0['volume_m3_perCap_2100_ssp2'], bins = volm3_bins, labels = volm3_labels, right=True)
df_0['range_len_m_2020'] = pd.cut(df_0['length_m_perCap_2020_ssp2'], bins = volm3_bins, labels = volm3_labels, right=True)
df_0['range_len_m_2050'] = pd.cut(df_0['length_m_perCap_2050_ssp2'], bins = volm3_bins, labels = volm3_labels, right=True)
df_0['range_len_m_2100'] = pd.cut(df_0['length_m_perCap_2100_ssp2'], bins = volm3_bins, labels = volm3_labels, right=True)

In [ ]:
from plotly.graph_objs import Line
import plotly.io as pio
import plotly.express as px

def plot_treemap(df, groupby_cols, color_column, filename=None):

    df = df[df[groupby_cols[0]] != 'noChange']

    df_infra_year = (
        df.groupby(groupby_cols, observed=False)
          .size()
          .reset_index(name='count')
    )

    fig = px.treemap(
        df_infra_year,
        path=groupby_cols,
        values="count",
        color=color_column,
        title="Load by city type and per capita range",
        color_discrete_map={
            'urban':'red',
            'suburban':'lightseagreen',
            'periurban':'dimgrey',
            'rural':'darkkhaki',
            'increasingLoad':'azure'
        }
    )

    fig.update_layout(
        width=600,
        height=600,
        autosize=False,
        uniformtext=dict(minsize=16)
    )

    fig.update_traces(
        marker_line_width=0.1,
        sort=False
    )

    fig.show()

    # Save interactive HTML (works in Colab)
    if filename is not None:
        html_name = filename.replace(".png", ".html")
        fig.write_html(html_name)
        print(f"Saved to {html_name}")

In [ ]:
df = df_0.copy()

color_col = 'city type'

group_cols = ['RBUV_Load_2050','city type','range_volm3_2050']
plot_treemap(df, group_cols, color_col, 'RBUV_2050.png')



Saved to RBUV_2050.html


In [ ]:
group_cols = ['RBUV_Load_2100','city type','range_volm3_2100']
plot_treemap(df, group_cols, color_col, 'RBUV_2100.png')


Saved to RBUV_2100.html


In [ ]:

group_cols = ['RL_Load_2050','city type','range_len_m_2050']
plot_treemap(df, group_cols, color_col, 'RL_2050.png')

group_cols = ['RL_Load_2100','city type','range_len_m_2100']
plot_treemap(df, group_cols, color_col, 'RL_2100.png')

Saved to RL_2050.html


Saved to RL_2100.html
